# Notebook 02 — Online Softmax, from First Principles

**FlashAttention from scratch · Apple M5 learning track**

**Objective:** understand and implement the numerically stable *streaming normalization* that allows FlashAttention to process attention scores in tiles, without holding an entire attention matrix.

**Prerequisites:** Python, NumPy, basic linear algebra, Notebook 01. No GPU required until the optional last section.

### What you will build
1. A naive softmax and a stable softmax.
2. A scalar, element-by-element online softmax algorithm.
3. An associative merge of softmax statistics for blocks of scores.
4. A *weighted* online softmax accumulator that directly produces an attention output.
5. Assertions for correctness, numerical stability and tile-size independence.

> This is an educational implementation. It demonstrates the mathematical core, **not yet a fused, optimized FlashAttention GPU kernel**. That comes in the later notebooks.

In [1]:
import math
import numpy as np

np.set_printoptions(precision=6, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.4


## 1. Why do we need softmax?

For one query vector $q\in\mathbb{R}^{d}$, and $N$ key/value rows,

$$s_j = \frac{q^\top k_j}{\sqrt d},\qquad
p_j=\frac{e^{s_j}}{\sum_{t=1}^N e^{s_t}},\qquad
o=\sum_{j=1}^N p_jv_j.$$

The scores $s_j$ can be arbitrarily large or negative. Softmax turns them into nonnegative weights that sum to one.

In normal attention, scores are arranged in an $N\times N$ matrix. FlashAttention avoids saving this entire matrix by working with **tiles**.

In [2]:
scores = np.array([1.0, 2.0, 3.0], dtype=np.float64)

def softmax_naive(x):
    exps = np.exp(x)
    return exps / np.sum(exps)

print("Scores:", scores)
print("Probabilities:", softmax_naive(scores))
print("Sum:", softmax_naive(scores).sum())
assert np.allclose(softmax_naive(scores).sum(), 1.0)

Scores: [1. 2. 3.]
Probabilities: [0.090031 0.244728 0.665241]
Sum: 1.0


## 2. Numerically stable softmax

Naively evaluating $e^{x_i}$ may overflow. Subtracting the maximum score preserves softmax exactly in real arithmetic:

$$\frac{e^{x_i}}{\sum_j e^{x_j}}
=\frac{e^{x_i-m}}{\sum_j e^{x_j-m}},\qquad m=\max_j x_j.$$

Because $x_i-m\leq 0$, the exponentials are at most 1, making overflow much less likely.

We will use two summary statistics:

$$m=\max_j x_j,\qquad \ell=\sum_j e^{x_j-m}.$$

Then $p_i=e^{x_i-m}/\ell$. The variable $\ell$ is sometimes called the *softmax denominator* (after max subtraction).

In [3]:
def softmax_stable(x):
    x = np.asarray(x)
    m = np.max(x)
    e = np.exp(x - m)
    return e / np.sum(e)

large_scores = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    bad = softmax_naive(large_scores)
print("Naive:", bad)
print("Stable:", softmax_stable(large_scores))
assert np.all(np.isfinite(softmax_stable(large_scores)))
assert np.allclose(softmax_stable(large_scores), softmax_stable(large_scores + 1e4))

Naive: [nan nan nan]
Stable: [0.090031 0.244728 0.665241]


## 3. Deriving streaming (online) softmax

Suppose we have seen scores $x_1,\ldots,x_t$. Store only:

$$m_t = \max_{1\le j\le t} x_j,\qquad
\ell_t = \sum_{j=1}^t e^{x_j-m_t}.$$

Now process one new score $x_{t+1}$:

$$m_{t+1}=\max(m_t,x_{t+1}).$$

The old sum is normalized with $m_t$, so when the maximum changes we **rescale it**:

$$\boxed{\ell_{t+1} = e^{m_t-m_{t+1}}\ell_t + e^{x_{t+1}-m_{t+1}}.}$$

**Proof:**

$$\begin{aligned}
\ell_{t+1}
&=\sum_{j=1}^{t+1} e^{x_j-m_{t+1}}\\
&= e^{m_t-m_{t+1}} \sum_{j=1}^{t} e^{x_j-m_t}
 + e^{x_{t+1}-m_{t+1}}\\
&= e^{m_t-m_{t+1}}\ell_t + e^{x_{t+1}-m_{t+1}}.
\end{aligned}$$

Start with $m_0=-\infty$ and $\ell_0=0$. Avoid explicitly computing $e^{-\infty-(-\infty)}$ when the state is empty. We assume a nonempty input and finite scores for this exercise.

In [4]:
def online_softmax_stats(x, trace=False):
    """One pass, O(1) running state. Returns (max, rescaled denominator)."""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or x.size == 0 or not np.all(np.isfinite(x)):
        raise ValueError("Expected a nonempty 1D array of finite scores")
    m, ell = -np.inf, 0.0
    history = []
    for score in x:
        new_m = max(m, float(score))
        old_scale = 0.0 if m == -np.inf else math.exp(m - new_m)
        ell = old_scale * ell + math.exp(float(score) - new_m)
        m = new_m
        history.append((float(score), m, ell))
    if trace:
        for t, (score, maximum, denominator) in enumerate(history, start=1):
            print(f"t={t} score={score:6.2f} m={maximum:6.2f} ell={denominator:.8f}")
    return m, ell

x = np.array([2.0, 1.0, 4.0, 3.0])
m, ell = online_softmax_stats(x, trace=True)
expected_m = np.max(x)
expected_ell = np.exp(x - expected_m).sum()
print("Online:", (m, ell), "Direct:", (expected_m, expected_ell))
assert np.allclose([m, ell], [expected_m, expected_ell], rtol=1e-14)

t=1 score=  2.00 m=  2.00 ell=1.00000000
t=2 score=  1.00 m=  2.00 ell=1.36787944
t=3 score=  4.00 m=  4.00 ell=1.18512235
t=4 score=  3.00 m=  4.00 ell=1.55300179
Online: (4.0, 1.553001792775919) Direct: (np.float64(4.0), np.float64(1.553001792775919))


### Important subtlety

The online algorithm determines the normalization constant in one pass. But to output **all individual probabilities** $p_j$, we generally still need the scores (or must recompute them), because the final $m$ and $\ell$ were not known when early elements were seen.

FlashAttention avoids this problem by accumulating the **weighted sum of values** directly, rather than producing the complete matrix of probabilities.

## 4. Processing a block at a time

For a block $B$ of scores and associated value vectors, define

$$m_B=\max_{j\in B}s_j,\quad
\ell_B=\sum_{j\in B}e^{s_j-m_B},\quad
a_B=\sum_{j\in B}e^{s_j-m_B}v_j.$$

If $A$ is the accumulated state from prior blocks, the merged statistics are:

$$m'=\max(m_A,m_B),$$
$$\ell'=e^{m_A-m'}\ell_A+e^{m_B-m'}\ell_B,$$
$$\boxed{a'=e^{m_A-m'}a_A+e^{m_B-m'}a_B.}$$

Here $a$ is a vector with the same dimension as $v_j$. At the end,

$$\boxed{o=a/\ell.}$$

**Why it works:** each previously accumulated contribution $e^{s_j-m_A}v_j$ is multiplied by $e^{m_A-m'}$, becoming $e^{s_j-m'}v_j$. The same is done for the new block. Thus every term uses the same final maximum.

The merge is associative in exact arithmetic (up to floating-point rounding in computers), so we can split the scores into blocks of different sizes and obtain the same mathematical result.

In [5]:
def block_summary(scores, values):
    """Summary (m, ell, a) of one nonempty block."""
    scores = np.asarray(scores, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    assert scores.ndim == 1 and scores.size > 0
    assert values.ndim == 2 and values.shape[0] == scores.size
    m = float(np.max(scores))
    w = np.exp(scores - m)
    ell = float(w.sum())
    a = w @ values
    return m, ell, a


def merge_summaries(A, B):
    """Combine two nonempty score/value summaries."""
    m_a, ell_a, a_a = A
    m_b, ell_b, a_b = B
    m = max(m_a, m_b)
    alpha = math.exp(m_a - m)
    beta = math.exp(m_b - m)
    return (m,
            alpha * ell_a + beta * ell_b,
            alpha * a_a + beta * a_b)

scores = np.array([2.0, 1.0, 4.0, 3.0])
values = np.array([[1., 0.], [0., 1.], [2., 1.], [1., 3.]])
A = block_summary(scores[:2], values[:2])
B = block_summary(scores[2:], values[2:])
merged = merge_summaries(A, B)
reference = softmax_stable(scores) @ values
print("Reference output:", reference)
print("Merged output:   ", merged[2] / merged[1])
assert np.allclose(merged[2] / merged[1], reference, atol=1e-14)

Reference output: [1.611856 1.386621]
Merged output:    [1.611856 1.386621]


## 5. Implement one-query attention, tile by tile

This function receives $s\in\mathbb R^N$ and $V\in\mathbb R^{N\times d_v}$. It never constructs a length-$N$ softmax probability vector; each iteration keeps only the current score tile and the running $m,\ell,a$.

Memory usage is $O(B+d_v)$ for the score/weight tile and running weighted result, apart from the provided inputs. The eventual GPU algorithm will **compute score tiles from $Q$ and $K$ on demand**, rather than receiving a complete score vector.

In [6]:
def tiled_weighted_softmax(scores, values, block_size):
    scores = np.asarray(scores, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    if scores.ndim != 1 or scores.size == 0 or values.ndim != 2 or values.shape[0] != scores.size:
        raise ValueError("Expected scores (N,) and values (N, d_v), with N > 0")
    if not np.all(np.isfinite(scores)) or not np.all(np.isfinite(values)):
        raise ValueError("Inputs must be finite")
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    state = None
    for start in range(0, scores.size, block_size):
        stop = min(start + block_size, scores.size)
        current = block_summary(scores[start:stop], values[start:stop])
        state = current if state is None else merge_summaries(state, current)
    m, ell, a = state
    return a / ell

rng = np.random.default_rng(2026)
scores = rng.normal(size=103)
values = rng.normal(size=(103, 16))
reference = softmax_stable(scores) @ values
for B in [1, 2, 7, 16, 32, 64, 103, 256]:
    actual = tiled_weighted_softmax(scores, values, B)
    error = np.max(np.abs(actual - reference))
    print(f"block size = {B:3d} | max absolute error = {error:.3e}")
    np.testing.assert_allclose(actual, reference, rtol=1e-12, atol=1e-12)

block size =   1 | max absolute error = 1.665e-16
block size =   2 | max absolute error = 3.886e-16
block size =   7 | max absolute error = 8.327e-17
block size =  16 | max absolute error = 1.665e-16
block size =  32 | max absolute error = 1.665e-16
block size =  64 | max absolute error = 1.006e-16
block size = 103 | max absolute error = 5.551e-17
block size = 256 | max absolute error = 5.551e-17


## 6. Connect it to scaled dot-product attention

For a single query, form the score tile only when you need it:

$$s_B = qK_B^\top / \sqrt d.$$

We can now write a small single-query version that **does not materialize the full attention-score vector**. It is a useful intermediate step toward a fused tiled kernel. In this notebook, NumPy operations may still allocate temporary arrays; the main saving is in the algorithmic *shape* of intermediates.

In [7]:
def attention_one_query_tiled(q, K, V, block_size):
    q, K, V = map(lambda x: np.asarray(x, dtype=np.float64), (q, K, V))
    if q.ndim != 1 or K.ndim != 2 or V.ndim != 2 or K.shape[1] != q.size or V.shape[0] != K.shape[0]:
        raise ValueError("Expected q (d,), K (N,d), V (N,d_v)")
    if K.shape[0] == 0 or block_size <= 0:
        raise ValueError("Expected N > 0 and block_size > 0")
    if not (np.isfinite(q).all() and np.isfinite(K).all() and np.isfinite(V).all()):
        raise ValueError("Inputs must be finite")
    state = None
    for start in range(0, K.shape[0], block_size):
        stop = min(start + block_size, K.shape[0])
        s_block = (K[start:stop] @ q) / math.sqrt(q.size)
        current = block_summary(s_block, V[start:stop])
        state = current if state is None else merge_summaries(state, current)
    return state[2] / state[1]

N, d, dv = 257, 32, 24
q = rng.normal(size=d)
K = rng.normal(size=(N, d))
V = rng.normal(size=(N, dv))
reference = softmax_stable((K @ q) / np.sqrt(d)) @ V
for B in [1, 8, 32, 64, 128, 300]:
    result = attention_one_query_tiled(q, K, V, B)
    print(f"B={B:3d}  error={np.max(np.abs(result-reference)):.3e}")
    np.testing.assert_allclose(result, reference, atol=1e-12, rtol=1e-12)

B=  1  error=1.353e-16
B=  8  error=7.633e-17
B= 32  error=4.163e-17
B= 64  error=5.204e-17
B=128  error=6.939e-17
B=300  error=5.898e-17


## 7. Numerical experiments

Test behavior when scores have a very large offset, and when one score dominates. The naive exponential expression may overflow or underflow; the stable blockwise algorithm should remain finite for finite input scores in these tests.

In [8]:
cases = {
    "large positive": np.array([1000., 1001., 999., 998.]),
    "large negative": np.array([-1000., -1001., -999., -998.]),
    "widely separated": np.array([-10000., -100., 0., 10000.]),
    "tied maxima": np.array([20., 20., 19., 20.]),
}
v = np.array([[1., 2.], [4., 3.], [2., 7.], [-1., 5.]])
for name, x in cases.items():
    expected = softmax_stable(x) @ v
    got = tiled_weighted_softmax(x, v, block_size=2)
    np.testing.assert_allclose(got, expected, atol=1e-12, rtol=1e-12)
    print(f"{name:20s} expected={expected} tiled={got}")

large positive       expected=[2.95477  3.175812] tiled=[2.95477  3.175812]
large negative       expected=[0.04523  5.148215] tiled=[0.04523  5.148215]
widely separated     expected=[-1.  5.] tiled=[-1.  5.]
tied maxima          expected=[1.406155 3.73385 ] tiled=[1.406155 3.73385 ]


## 8. Optional: confirm PyTorch MPS on your Apple M5

This is only a **device and reference comparison**, not our custom GPU kernel. PyTorch's `torch.softmax` executes using supported MPS operations. NumPy reference uses float64; we compare using float32 with a reasonable tolerance for different arithmetic and reduction order.

> **Note:** Apple GPU/MPS precision and supported operations may differ from CPU; in future notebooks we will explicitly measure tolerances for each dtype.

In [9]:
try:
    import torch
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        s_t = torch.tensor(scores, dtype=torch.float32, device=device)
        v_t = torch.tensor(values, dtype=torch.float32, device=device)
        out_t = torch.softmax(s_t, dim=0) @ v_t
        out_np = tiled_weighted_softmax(scores, values, block_size=17)
        print("MPS:", out_t.device, "PyTorch:", torch.__version__)
        print("MPS vs NumPy max error:", np.max(np.abs(out_t.cpu().numpy() - out_np)))
        np.testing.assert_allclose(out_t.cpu().numpy(), out_np, rtol=1e-5, atol=1e-5)
    else:
        print("MPS unavailable; CPU/NumPy sections above are sufficient.")
except ImportError:
    print("PyTorch not installed; optional MPS check skipped.")

MPS: mps:0 PyTorch: 2.14.1
MPS vs NumPy max error: 4.5097322154408204e-08


## 9. Exercises (recommended before Notebook 03)

**Exercise A — Derive the scalar update yourself.** Prove that changing $m$ requires multiplying the old $\ell$ by $e^{m_{\text{old}}-m_{\text{new}}}$.

**Exercise B — Test associativity.** Split a score/value array into three pieces, compute summaries $A,B,C$, and verify that `merge(merge(A,B),C)` and `merge(A,merge(B,C))` agree to floating-point tolerance.

**Exercise C — Force the maximum to change late.** Use scores `[-50, -20, 3, 90]`, print $(m,\ell,a)$ for each tile, and explain why the old accumulator is rescaled.

**Exercise D — Measure memory conceptually.** Standard self-attention has $N^2$ scores. In a tiled implementation processing $B_Q$ query rows and $B_K$ key rows, an individual score tile has $B_QB_K$ elements. Write down the ratio to $N^2$.

**Exercise E — Extend the code.** Change the one-query implementation to support an additive attention mask. What should happen if every score in a tile is masked? (Hint: the block should contribute nothing; handle the $-\infty$ case carefully.)

### Stop and check

You should be able to explain, without looking at the code:
- why ordinary `exp(x)` can overflow,
- why online softmax needs a running maximum,
- why both $\ell$ **and** $a$ are rescaled on a maximum update,
- why the final output is $a/\ell$,
- why this removes the need to store all attention weights.

**Next:** Notebook 03 — process *multiple query rows* at once, implement the original FlashAttention-1 forward tiling algorithm, compare with standard attention, and inspect memory scaling.